## Anomaly Detection & Misuse Detection | NSL-KDD Dataset

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import auc, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

COLUMNS = [
    'duration','protocol_type','service','flag',
    'src_bytes','dst_bytes','land','wrong_fragment','urgent','hot',
    'num_failed_logins','logged_in','num_compromised','root_shell',
    'su_attempted','num_root','num_file_creations','num_shells',
    'num_access_files','num_outbound_cmds','is_host_login','is_guest_login',
    'count','srv_count','serror_rate','srv_serror_rate','rerror_rate',
    'srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate',
    'dst_host_count','dst_host_srv_count','dst_host_same_srv_rate',
    'dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate',
    'dst_host_srv_rerror_rate','label','difficulty'
]

# Three symbolic (nominal/categorical) columns removed for distance calculation (mentioned in the assignment)
SYMBOLIC_COLS = ['protocol_type', 'service', 'flag']

#Difficulty is a meta-label indicating difficulty in classification, it's not considered for prediction as it's not really a feature (mentioned in the assignment)
NUMERIC_COLS  = [c for c in COLUMNS if c not in SYMBOLIC_COLS + ['label', 'difficulty']]

ATTACK_MAP = {
    'back':'dos','land':'dos','neptune':'dos','pod':'dos','smurf':'dos','teardrop':'dos',
    'ipsweep':'probe','nmap':'probe','portsweep':'probe','satan':'probe',
    'ftp_write':'r2l','guess_passwd':'r2l','imap':'r2l','multihop':'r2l',
    'phf':'r2l','spy':'r2l','warezclient':'r2l','warezmaster':'r2l',
    'buffer_overflow':'u2r','loadmodule':'u2r','perl':'u2r','rootkit':'u2r',
    'normal':'normal',
}
def map_attack(label):
    return ATTACK_MAP.get(label.strip().lower(), 'unknown')

CATEGORIES  = ['normal', 'probe', 'dos', 'u2r', 'r2l']
cat_to_idx  = {c: i for i, c in enumerate(CATEGORIES)}

COST_MATRIX = np.array([
    [0, 1, 2, 2, 2],
    [1, 0, 2, 2, 2],
    [2, 2, 0, 2, 2],
    [3, 3, 3, 0, 3],
    [4, 4, 4, 4, 0],
])

cost_matrix_df = pd.DataFrame(COST_MATRIX, index=CATEGORIES, columns=CATEGORIES)

train_df = pd.read_csv('20 Percent Training Set.csv', header=None, names=COLUMNS)

#KDDTest+.csv in the github is malformed (missing newlines), so we've used the equivalent .txt file.
test_df  = pd.read_csv('KDDTest+.txt', header=None, names=COLUMNS)
train_df.drop(columns=['difficulty'], inplace=True)
test_df.drop(columns=['difficulty'],  inplace=True)

X_test_raw    = test_df[NUMERIC_COLS].values
y_test_binary = (test_df['label'] != 'normal').astype(int).values

print(f"Training: {len(train_df):,} records | Test: {len(test_df):,} records")
print(f"Numeric features used: {len(NUMERIC_COLS)}")
print("\n")
print("Cost Matrix:")
display(cost_matrix_df)

FileNotFoundError: [Errno 2] No such file or directory: '20 Percent Training Set.csv'

---
# Part A: Anomaly Detection

## A(1) — Extract normal instances; normalize attribute values

A(3)

In [ ]:
train_normal       = train_df[train_df['label'] == 'normal']
X_train_normal_raw = train_normal[NUMERIC_COLS].values

scaler_A       = MinMaxScaler()
X_train_normal = scaler_A.fit_transform(X_train_normal_raw)
X_test_A       = scaler_A.transform(X_test_raw)

print(f"Normal training instances : {len(train_normal):,}")
print(f"Scaler range per feature  : [0, 1]  (Min-Max normalization)")


NameError: name 'train_df' is not defined

## A(2) — Find nearest normal neighbor & compute distance for each test instance

In [ ]:
nn_model = NearestNeighbors(n_neighbors=1, algorithm='ball_tree', metric='euclidean', n_jobs=-1)
nn_model.fit(X_train_normal)
distances, _           = nn_model.kneighbors(X_test_A)
dist_to_nearest_normal = distances[:, 0]

pd.Series(dist_to_nearest_normal).describe().rename('Distance to Nearest Normal Neighbor')


A(3) - Varying the control threshold; Classifying each new instance as normal or attack

In [ ]:
#creating the thresholds
thresholds = np.linspace(
    dist_to_nearest_normal.min(),
    dist_to_nearest_normal.max(),
    10
)

#making the predictions
predictions_dict = {}

for t in thresholds:
    predictions_dict[t] = [
        0 if d <= t else 1
        for d in dist_to_nearest_normal
    ]

# Part B: Misuse Detection

## B(1) - Use the whole training dataset and normalize